# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrinidhi-02/flyrank-machine-learning/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Shrinidhi-02/flyrank-machine-learning/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Clients:", df["client_id"].nunique())
print("Required outcome exists:", "trend_direction" in df.columns)

display(df.head())

Dataset loaded successfully
Rows: 30000
Columns: 44
Clients: 32
Required outcome exists: True


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


I will use Logistic Regression as my first model because the task is to identify content items whose observed trend is declining. Logistic Regression is a simple classification method that provides a useful starting point before trying more complex models.

The model will use content and search-context features. I will exclude `trend_direction` and `trend_pct` from the input features because they are related to the outcome label. I will also exclude content and client IDs because they are identifiers, not meaningful predictive features.

I will compare the model with the existing rule-based baseline on held-out clients. The goal is to measure whether the model ranks declining items more usefully, not to assume that a more complex model is automatically better.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the observed outcome label.
df = df.copy()
df["target_declining"] = (
    df["trend_direction"]
    .astype("string")
    .str.lower()
    .eq("down")
    .astype(int)
)

# Remove rows without a usable client ID or target.
df = df.dropna(subset=["client_id", "target_declining"]).copy()

# Keep all rows belonging to a client together.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())
print(
    "Client overlap:",
    len(set(train_df["client_id"]) & set(test_df["client_id"]))
)
print("Training decline rate:", round(train_df["target_declining"].mean(), 4))
print("Test decline rate:", round(test_df["target_declining"].mean(), 4))

assert set(train_df["client_id"]).isdisjoint(set(test_df["client_id"]))
assert train_df["target_declining"].nunique() == 2

print("Grouped split check passed.")

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0
Training decline rate: 0.5501
Test decline rate: 0.511
Grouped split check passed.


I will split the data by `client_id`, keeping each client's content entirely in either the training set or the test set. This is more appropriate than a random row split because the dataset contains multiple clients, and a row-level split could place content from the same client in both sets.

I will use a fixed random seed for reproducibility. The model will learn from the training clients and be evaluated on held-out clients. I will compare it with the rule-based baseline on exactly the same test rows.

The target will be 1 when `trend_direction` is `down`, and 0 otherwise. This is an analysis of the observed decline label, not proof that the model predicts future performance.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Features chosen to avoid direct use of the outcome fields
# and recent trend calculations.
numeric_features = [
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
]

feature_columns = numeric_features + categorical_features

missing_features = [
    col for col in feature_columns if col not in df.columns
]
if missing_features:
    raise ValueError(f"Missing expected features: {missing_features}")

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df["target_declining"]
y_test = test_df["target_declining"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )),
])

model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]

print("Logistic Regression trained successfully.")
print("Test predictions:", len(model_scores))


# Recreate the Week 4 rule using training-set thresholds only.
refresh_threshold = train_df["days_since_last_update"].quantile(0.75)
volume_threshold = train_df["impressions_90d"].quantile(0.75)

valid_train_top10 = (
    train_df["avg_position"].notna()
    & (train_df["avg_position"] > 0)
    & (train_df["avg_position"] <= 10)
    & train_df["ctr"].notna()
)

if valid_train_top10.sum() == 0:
    raise ValueError("No valid top-10 training rows for the CTR threshold.")

top10_ctr_median = train_df.loc[valid_train_top10, "ctr"].median()

refresh_flag = (
    test_df["days_since_last_update"].notna()
    & (test_df["days_since_last_update"] >= refresh_threshold)
)

ctr_flag = (
    test_df["avg_position"].notna()
    & (test_df["avg_position"] > 0)
    & (test_df["avg_position"] <= 10)
    & test_df["ctr"].notna()
    & (test_df["ctr"] < top10_ctr_median)
)

volume_flag = (
    test_df["impressions_90d"].notna()
    & (test_df["impressions_90d"] >= volume_threshold)
)

baseline_scores = (
    refresh_flag.astype(int) * 2
    + ctr_flag.astype(int) * 2
    + volume_flag.astype(int)
).to_numpy()


def precision_at_k(labels, scores, k):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    k = min(k, len(labels))
    order = np.argsort(-scores, kind="stable")[:k]
    return labels[order].mean()


def average_precision(labels, scores):
    if len(np.unique(labels)) < 2:
        return np.nan
    return average_precision_score(labels, scores)


comparison = pd.DataFrame([
    {
        "Approach": "Week 4 rule baseline",
        "Precision@20": precision_at_k(y_test, baseline_scores, 20),
        "Precision@50": precision_at_k(y_test, baseline_scores, 50),
        "Average precision": average_precision(y_test, baseline_scores),
    },
    {
        "Approach": "Logistic Regression",
        "Precision@20": precision_at_k(y_test, model_scores, 20),
        "Precision@50": precision_at_k(y_test, model_scores, 50),
        "Average precision": average_precision(y_test, model_scores),
    },
])

print("Test-set decline rate (base rate):", round(y_test.mean(), 4))
display(comparison.round(4))

Logistic Regression trained successfully.
Test predictions: 6163
Test-set decline rate (base rate): 0.511


,Approach,Precision@20,Precision@50,Average precision
0,Week 4 rule baseline,0.65,0.62,0.52
1,Logistic Regression,0.30,0.38,0.51


I will train Logistic Regression using content and search-context fields, with missing values handled by imputers and categorical values encoded using one-hot encoding. All preprocessing will be fitted using training data only.

The baseline will reproduce the Week 4 rule: two points for an old update, two points for low CTR among valid top-10 results, and one point for high impression volume. I will calculate its thresholds using training data and apply them to the test set.

I will compare both approaches on the same held-out rows using precision@20, precision@50, average precision, and the test-set decline rate. Precision@K measures the fraction of the top K ranked items that have the observed decline label. These results are decision-support evidence, not proof of causation.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create a readable error-analysis table.
error_analysis = test_df[
    ["content_id", "client_id", "target_declining"]
].copy()

error_analysis["model_score"] = model_scores
error_analysis["baseline_score"] = baseline_scores
error_analysis["model_prediction"] = (model_scores >= 0.5).astype(int)

error_analysis["error_type"] = "Correct"
error_analysis.loc[
    (error_analysis["model_prediction"] == 1)
    & (error_analysis["target_declining"] == 0),
    "error_type"
] = "False positive"

error_analysis.loc[
    (error_analysis["model_prediction"] == 0)
    & (error_analysis["target_declining"] == 1),
    "error_type"
] = "False negative"

print("Error counts:")
display(error_analysis["error_type"].value_counts().to_frame("count"))

print("\nExamples of model errors:")
errors = error_analysis[
    error_analysis["error_type"] != "Correct"
].sort_values("model_score", ascending=False)

display(errors.head(10))

# Permutation importance on held-out clients.
importance = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=3,
    random_state=42,
)

importance_table = pd.DataFrame({
    "feature": feature_columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False)

print("\nPermutation importance (top features):")
display(importance_table.head(10).round(4))

print("\nFinal checks:")
print("Same test rows for both approaches:", len(baseline_scores) == len(model_scores))
print("Test rows:", len(test_df))
print("Baseline precision@20:", round(precision_at_k(y_test, baseline_scores, 20), 4))
print("Model precision@20:", round(precision_at_k(y_test, model_scores, 20), 4))

Error counts:


,count
error_type,
Correct,3199
False positive,1528
False negative,1436



Examples of model errors:


,content_id,client_id,target_declining,model_score,baseline_score,model_prediction,error_type
14300,content_24b8d8860ad9,client_8527a891e2,0,0.788589,2,1,False positive
9832,content_45b7ba1087e0,client_8527a891e2,0,0.778532,0,1,False positive
22079,content_881635edbf8b,client_8527a891e2,0,0.778207,0,1,False positive
29091,content_fc48c1737c57,client_8527a891e2,0,0.777264,0,1,False positive
9776,content_c5f79bf532f3,client_8527a891e2,0,0.776495,2,1,False positive
28938,content_0ae004d4a60f,client_8527a891e2,0,0.776369,0,1,False positive
558,content_58c4b92b92aa,client_8527a891e2,0,0.776301,2,1,False positive
20439,content_ea8a83a2510c,client_8527a891e2,0,0.776164,0,1,False positive
27164,content_97ec8ce69833,client_8527a891e2,0,0.776138,0,1,False positive
5394,content_38bab00f71e2,client_8527a891e2,0,0.776061,2,1,False positive



Permutation importance (top features):


,feature,importance_mean,importance_std
0,content_age_days,0.0216,0.0032
9,competition_level,0.0013,0.0003
5,word_count,0.0011,0.0007
4,cpc,0.0005,0.0005
2,search_volume,0.0005,0.0004
7,content_type,0.0000,0.0000
10,age_tier,-0.0013,0.0019
11,freshness_tier,-0.0014,0.0001
3,competition,-0.0018,0.0002
12,word_count_tier,-0.0022,0.0005



Final checks:
Same test rows for both approaches: True
Test rows: 6163
Baseline precision@20: 0.65
Model precision@20: 0.3


I will inspect false positives and false negatives to understand where the model makes mistakes. A false positive is an item predicted as declining that does not have the observed decline label. A false negative is a declining item the model misses.

I will also inspect permutation importance to see which input features contribute most to the model's average precision. Feature importance describes model behaviour, not causation. I will use the actual results to discuss limitations and avoid claiming that the model predicts future improvements.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Conclusion

I evaluated Logistic Regression against the Week 4 rule-based baseline using a grouped client split. The training set contained 23,837 rows from 25 clients, and the test set contained 6,163 rows from 7 different clients, with zero client overlap.

The baseline achieved Precision@20 of 0.65 and Precision@50 of 0.62. Logistic Regression achieved Precision@20 of 0.30 and Precision@50 of 0.38. Average precision was 0.52 for the baseline and 0.51 for Logistic Regression.

On this held-out test set, the baseline performed better at identifying declining items near the top of the ranking. Logistic Regression did not improve the results enough to justify replacing the simpler rule. Content age had the highest measured permutation importance in the model, although feature importance does not establish causation.

These results are limited to the observed decline label and this particular client split. The model and baseline provide decision-support evidence, not proof of future performance or causal effects.